# Experiment: Project Bootstrap And ETIS Data Inspection

本 notebook 只负责 ETIS 项目的准备与检查：

- 固定项目目标和运行顺序
- 检查你已经放好的 ETIS 目录和列表文件
- 确认 `train / val / test = 156 / 20 / 20`
- 固定后续所有模型共用的测试可视化样本


## 加载基础工具

这里仅加载最基础的环境、目录和 JSON 工具，不包含模型与训练逻辑。


In [ ]:
from pathlib import Path
import json
import os
import random
import sys

from scripts.project_utils import (
    ARTIFACTS,
    DATA_ROOT,
    ETIS_ROOT,
    PVT_PRETRAINED_ROOT,
    ensure_project_dirs,
    get_default_project_config,
    load_project_config,
    print_env_summary,
    save_json,
    set_seed,
    try_import_torch,
)

ensure_project_dirs()
set_seed()
torch = try_import_torch()
ENV_SUMMARY = print_env_summary(torch)
ROOT = Path.cwd().resolve()


## 保存统一项目配置

这一段把 ETIS 数据路径、预训练权重路径、训练超参数和统一输出目录写入配置文件，供后续所有 notebook 共用。


In [ ]:
PROJECT_CONFIG = get_default_project_config()
save_json(PROJECT_CONFIG, ARTIFACTS / "project_config.json")
PROJECT_CONFIG


## 定义 ETIS 数据检查函数

这里直接针对你当前的 ETIS 目录结构进行检查：

- `train/images` 与 `train/masks`
- `val/images` 与 `val/masks`
- `test/images` 与 `test/masks`
- 三个 list 文件


In [ ]:
assert torch is not None, "需要先安装 PyTorch 才能运行本 notebook。"

from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

ETIS_SPLITS = {
    "train": ETIS_ROOT / "train",
    "val": ETIS_ROOT / "val",
    "test": ETIS_ROOT / "test",
}

def read_list_file(path):
    return [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

def collect_split_summary(split_name):
    split_root = ETIS_SPLITS[split_name]
    image_dir = split_root / "images"
    mask_dir = split_root / "masks"
    images = sorted(p.name for p in image_dir.glob("*") if p.is_file())
    masks = sorted(p.name for p in mask_dir.glob("*") if p.is_file())
    list_file = ETIS_ROOT / f"{split_name}_list_etis.txt"
    listed = read_list_file(list_file)
    return {
        "split": split_name,
        "image_count": len(images),
        "mask_count": len(masks),
        "list_count": len(listed),
        "image_mask_match": images == masks,
        "list_match_images": listed == images,
        "preview": images[:8],
    }

def infer_fixed_visual_sample():
    test_items = read_list_file(ETIS_ROOT / "test_list_etis.txt")
    return test_items[0] if test_items else None

def load_rgb_mask_pair(split_name, filename):
    image_path = ETIS_SPLITS[split_name] / "images" / filename
    mask_path = ETIS_SPLITS[split_name] / "masks" / filename
    image = np.array(Image.open(image_path).convert("RGB"))
    mask = np.array(Image.open(mask_path).convert("L"))
    return image, mask

def visualize_pair(split_name, filename):
    image, mask = load_rgb_mask_pair(split_name, filename)
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(image)
    axes[0].set_title(f"{split_name} image")
    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("mask")
    for ax in axes:
        ax.axis("off")
    fig.tight_layout()
    return fig


## 执行 ETIS 数据检查

这一段会验证每个 split 的图片数、mask 数、列表数是否一致，并确定后续统一可视化的测试样本。


In [ ]:
split_summaries = {split: collect_split_summary(split) for split in ["train", "val", "test"]}
fixed_visual_sample = infer_fixed_visual_sample()
PROJECT_CONFIG["fixed_visual_sample"] = fixed_visual_sample
save_json(PROJECT_CONFIG, ARTIFACTS / "project_config.json")
split_summaries, fixed_visual_sample


## 可视化一个真实样本

这里展示固定测试样本的原图和标注，方便后续所有模型做统一对比。


In [ ]:
fig = visualize_pair("test", fixed_visual_sample)
fig


## 保存 bootstrap 检查结果

这里把 ETIS 当前划分和统一可视化样本写入 records，作为后续实验的基础记录。


In [ ]:
save_json(
    {
        "env_summary": ENV_SUMMARY,
        "dataset": "ETIS",
        "split_summaries": split_summaries,
        "fixed_visual_sample": fixed_visual_sample,
        "pretrained_path": PROJECT_CONFIG["pvt_pretrained_path"],
    },
    ARTIFACTS / "records" / "bootstrap_etis_summary.json",
)
